# Introducción a Pandas: manipulación y limpieza de datos

## Objetivo
En esta notebook vamos a trabajar con un **dataset sintético de clientes** que contiene varios problemas habituales de calidad de datos.

La idea es aprender a:

- Crear y cargar un `DataFrame`.
- Inspeccionar filas, columnas, tipos y estadísticas.
- Seleccionar y filtrar datos.
- Ordenar y contar valores.
- Detectar valores faltantes.
- Detectar registros duplicados.
- Corregir categorías inconsistentes.
- Convertir tipos de datos.
- Detectar valores imposibles y outliers.
- Reemplazar e imputar valores.
- Crear y eliminar columnas.
- Agrupar y resumir información.
- Dejar un dataset limpio para una etapa posterior de Machine Learning.

> **Importante:** los datos son ficticios y fueron diseñados intencionalmente con errores para practicar limpieza.

## 1. Importar las bibliotecas
`pandas` será la biblioteca principal. También utilizaremos `numpy` para representar valores faltantes y generar datos.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Crear datos sintéticos

Generaremos clientes con las siguientes variables:

- `id_cliente`
- `edad`
- `provincia`
- `ingresos`
- `antiguedad_anios`
- `compras_anuales`
- `satisfaccion`
- `fecha_alta`
- `activo`

Luego introduciremos deliberadamente **faltantes, duplicados, categorías inconsistentes, tipos incorrectos y valores extremos**.

In [ ]:
np.random.seed(42)
n = 100

df = pd.DataFrame({
    "id_cliente": range(1001, 1001 + n),
    "edad": np.random.randint(18, 70, n).astype(float),
    "provincia": np.random.choice(
        ["Tierra del Fuego", "Buenos Aires", "Córdoba", "Santa Fe"],
        n,
        p=[0.25, 0.35, 0.20, 0.20]
    ),
    "ingresos": np.random.normal(950_000, 280_000, n).round(0),
    "antiguedad_anios": np.random.randint(0, 16, n),
    "compras_anuales": np.random.randint(0, 35, n),
    "satisfaccion": np.random.randint(1, 6, n).astype(float),
    "fecha_alta": pd.date_range("2021-01-01", periods=n, freq="12D").astype(str),
    "activo": np.random.choice(["Sí", "No"], n, p=[0.8, 0.2])
})

# Introducimos problemas intencionalmente
df.loc[[3, 17, 41], "edad"] = np.nan                    # faltantes
df.loc[[8, 29], "ingresos"] = np.nan                   # faltantes
df.loc[[12, 55], "satisfaccion"] = np.nan              # faltantes

df.loc[5, "edad"] = -4                                 # valor imposible
df.loc[21, "edad"] = 250                               # valor imposible
df.loc[33, "ingresos"] = 12_000_000                   # outlier
df.loc[72, "ingresos"] = -300_000                     # valor imposible

df.loc[10, "provincia"] = "tierra del fuego"           # categoría inconsistente
df.loc[20, "provincia"] = "TDF"
df.loc[30, "provincia"] = "Buenos aires"
df.loc[40, "provincia"] = "CORDOBA"

df.loc[15, "activo"] = "SI"
df.loc[25, "activo"] = "si"
df.loc[35, "activo"] = "NO"

df.loc[45, "fecha_alta"] = "fecha_desconocida"         # fecha inválida

# Agregamos dos duplicados completos
df = pd.concat([df, df.iloc[[7, 18]]], ignore_index=True)

df.head(10)

## 3. Primera inspección del DataFrame

Antes de modificar datos conviene **conocer el dataset**.

In [ ]:
print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

In [ ]:
# Primeras filas
df.head()

In [ ]:
# Últimas filas
df.tail()

In [ ]:
# Muestra aleatoria
df.sample(5, random_state=1)

In [ ]:
# Información general: tipos de datos, cantidad de valores no nulos y memoria
df.info()

In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe()

In [ ]:
# Estadísticas incluyendo variables categóricas
df.describe(include="all")

## 4. Selección de filas y columnas

Pandas permite seleccionar columnas individuales, varias columnas y filas específicas.

In [ ]:
# Una columna
df["edad"].head()

In [ ]:
# Varias columnas
df[["edad", "provincia", "ingresos"]].head()

In [ ]:
# Selección por etiquetas con .loc
df.loc[0:5, ["id_cliente", "edad", "provincia"]]

In [ ]:
# Selección por posición con .iloc
df.iloc[0:5, 0:4]

## 5. Filtrado de datos

Podemos seleccionar solamente las observaciones que cumplen una condición.

In [ ]:
# Clientes mayores de 50 años
df[df["edad"] > 50].head()

In [ ]:
# Varias condiciones
df[(df["edad"] > 40) & (df["ingresos"] > 1_000_000)].head()

In [ ]:
# Filtrar por categorías
df[df["provincia"].isin(["Tierra del Fuego", "Buenos Aires"])].head()

## 6. Ordenamiento, frecuencias y valores únicos
Estas operaciones son muy útiles durante la inspección inicial.

In [ ]:
# Ordenar por ingresos de mayor a menor
df.sort_values("ingresos", ascending=False).head(10)

In [ ]:
# Categorías existentes
df["provincia"].unique()

In [ ]:
# Cantidad de categorías diferentes
df["provincia"].nunique()

In [ ]:
# Frecuencia de cada categoría
df["provincia"].value_counts(dropna=False)

# PARTE II — Limpieza de datos
Ahora vamos a buscar y corregir los problemas que podrían afectar a un futuro modelo.

## 7. Detectar valores faltantes

In [ ]:
# Cantidad de valores faltantes por columna
df.isnull().sum()

In [ ]:
# Porcentaje de valores faltantes
(df.isnull().mean() * 100).round(2)

In [ ]:
# Filas que contienen al menos un valor faltante
df[df.isnull().any(axis=1)]

### ¿Qué podemos hacer con los faltantes?

No existe una única solución. Dependiendo del problema podemos:

- eliminar filas;
- eliminar columnas;
- completar con media o mediana;
- completar con moda;
- utilizar métodos de imputación más avanzados.

En este ejemplo imputaremos `edad` e `ingresos` con la **mediana** y `satisfaccion` con la **moda**.

In [ ]:
df_limpio = df.copy()

df_limpio["edad"] = df_limpio["edad"].fillna(df_limpio["edad"].median())
df_limpio["ingresos"] = df_limpio["ingresos"].fillna(df_limpio["ingresos"].median())
df_limpio["satisfaccion"] = df_limpio["satisfaccion"].fillna(df_limpio["satisfaccion"].mode()[0])

df_limpio.isnull().sum()

## 8. Detectar y eliminar duplicados

In [ ]:
print("Cantidad de filas duplicadas:", df_limpio.duplicated().sum())

df_limpio[df_limpio.duplicated(keep=False)].sort_values("id_cliente")

In [ ]:
df_limpio = df_limpio.drop_duplicates().copy()
print("Nuevas dimensiones:", df_limpio.shape)

## 9. Corregir categorías inconsistentes

Una misma categoría escrita de varias formas puede ser interpretada como categorías diferentes.

In [ ]:
print(df_limpio["provincia"].value_counts())

In [ ]:
reemplazos_provincia = {
    "tierra del fuego": "Tierra del Fuego",
    "TDF": "Tierra del Fuego",
    "Buenos aires": "Buenos Aires",
    "CORDOBA": "Córdoba"
}

df_limpio["provincia"] = df_limpio["provincia"].replace(reemplazos_provincia)

df_limpio["provincia"].value_counts()

In [ ]:
# Estandarizamos también la variable activo
df_limpio["activo"] = df_limpio["activo"].replace({
    "SI": "Sí",
    "si": "Sí",
    "NO": "No"
})

df_limpio["activo"].value_counts()

## 10. Convertir tipos de datos

`fecha_alta` fue creada como texto. Para trabajar correctamente con fechas conviene convertirla a `datetime`.

Usaremos `errors="coerce"` para transformar fechas inválidas en `NaT` (valor faltante de fecha).

In [ ]:
df_limpio["fecha_alta"] = pd.to_datetime(df_limpio["fecha_alta"], errors="coerce")

df_limpio.info()

In [ ]:
# Localizamos las fechas que no pudieron convertirse
df_limpio[df_limpio["fecha_alta"].isna()]

In [ ]:
# En este ejercicio eliminamos el registro con fecha inválida.
# En un caso real habría que investigar el dato antes de decidir.
df_limpio = df_limpio.dropna(subset=["fecha_alta"]).copy()

## 11. Buscar valores imposibles

Los estadísticos mínimos y máximos pueden revelar errores rápidamente.

In [ ]:
df_limpio[["edad", "ingresos", "antiguedad_anios", "compras_anuales", "satisfaccion"]].describe()

In [ ]:
# Edades fuera de un rango razonable para este ejemplo
df_limpio[(df_limpio["edad"] < 18) | (df_limpio["edad"] > 100)]

In [ ]:
# Ingresos negativos
df_limpio[df_limpio["ingresos"] < 0]

En este ejercicio consideraremos esos valores como errores y los convertiremos primero en `NaN`, para luego imputarlos.

In [ ]:
df_limpio.loc[(df_limpio["edad"] < 18) | (df_limpio["edad"] > 100), "edad"] = np.nan
df_limpio.loc[df_limpio["ingresos"] < 0, "ingresos"] = np.nan

df_limpio["edad"] = df_limpio["edad"].fillna(df_limpio["edad"].median())
df_limpio["ingresos"] = df_limpio["ingresos"].fillna(df_limpio["ingresos"].median())

## 12. Detectar outliers con el rango intercuartílico (IQR)

Un outlier **no necesariamente es un error**. Debe investigarse antes de eliminarlo.

Usaremos el criterio:

**Límite inferior = Q1 − 1.5 × IQR**  
**Límite superior = Q3 + 1.5 × IQR**

In [ ]:
Q1 = df_limpio["ingresos"].quantile(0.25)
Q3 = df_limpio["ingresos"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)

outliers_ingresos = df_limpio[
    (df_limpio["ingresos"] < limite_inferior) |
    (df_limpio["ingresos"] > limite_superior)
]

outliers_ingresos

Para fines didácticos, eliminaremos los outliers de ingresos.  
**En un proyecto real no deberían eliminarse automáticamente:** primero debemos determinar si son errores o casos reales.

In [ ]:
df_limpio = df_limpio[
    (df_limpio["ingresos"] >= limite_inferior) &
    (df_limpio["ingresos"] <= limite_superior)
].copy()

## 13. Crear nuevas columnas

Pandas también permite transformar variables y generar información derivada.

In [ ]:
# Crear una categoría de edad
df_limpio["grupo_edad"] = pd.cut(
    df_limpio["edad"],
    bins=[17, 30, 45, 60, 100],
    labels=["18-30", "31-45", "46-60", "61+"]
)

# Ingreso mensual expresado en miles
df_limpio["ingresos_miles"] = (df_limpio["ingresos"] / 1000).round(1)

df_limpio.head()

## 14. Renombrar y eliminar columnas

In [ ]:
# Ejemplo de renombrado
df_limpio = df_limpio.rename(columns={
    "antiguedad_anios": "antiguedad"
})

# Eliminamos una columna derivada que ya no necesitamos
df_limpio = df_limpio.drop(columns=["ingresos_miles"])

df_limpio.head()

## 15. Agrupar y resumir datos con `groupby`

`groupby()` permite responder preguntas como:

- ¿Cuál es el ingreso promedio por provincia?
- ¿Cuál es la edad promedio?
- ¿Cuántas compras realizan?
- ¿Cuántos clientes hay en cada grupo?

In [ ]:
resumen_provincia = (
    df_limpio
    .groupby("provincia")
    .agg(
        cantidad_clientes=("id_cliente", "count"),
        edad_promedio=("edad", "mean"),
        ingreso_promedio=("ingresos", "mean"),
        compras_promedio=("compras_anuales", "mean")
    )
    .round(2)
    .sort_values("ingreso_promedio", ascending=False)
)

resumen_provincia

## 16. Verificación final de calidad

Después de limpiar, volvemos a inspeccionar. La limpieza no termina hasta verificar el resultado.

In [ ]:
print("Dimensiones finales:", df_limpio.shape)
print("\nValores faltantes:")
print(df_limpio.isnull().sum())

print("\nDuplicados:", df_limpio.duplicated().sum())

print("\nProvincias:")
print(df_limpio["provincia"].value_counts())

print("\nActivo:")
print(df_limpio["activo"].value_counts())

In [ ]:
df_limpio.describe(include="all")

## 17. Guardar el dataset limpio

Una vez finalizada la preparación podemos exportar el resultado para utilizarlo posteriormente.

In [ ]:
df_limpio.to_csv("clientes_limpios.csv", index=False, encoding="utf-8-sig")

print("Archivo clientes_limpios.csv generado correctamente.")

# Resumen de funciones utilizadas

| Función / atributo | Uso |
|---|---|
| `pd.DataFrame()` | Crear un DataFrame |
| `head()` / `tail()` | Ver primeras/últimas filas |
| `sample()` | Obtener una muestra |
| `shape` | Dimensiones |
| `columns` | Nombres de columnas |
| `info()` | Tipos y valores no nulos |
| `describe()` | Estadísticas descriptivas |
| `loc[]` / `iloc[]` | Seleccionar datos |
| `isin()` | Filtrar por varios valores |
| `sort_values()` | Ordenar |
| `unique()` | Valores únicos |
| `nunique()` | Cantidad de valores únicos |
| `value_counts()` | Frecuencias |
| `isnull()` | Detectar faltantes |
| `fillna()` | Imputar faltantes |
| `duplicated()` | Detectar duplicados |
| `drop_duplicates()` | Eliminar duplicados |
| `replace()` | Reemplazar valores |
| `pd.to_datetime()` | Convertir fechas |
| `quantile()` | Calcular cuantiles |
| `pd.cut()` | Crear intervalos/categorías |
| `rename()` | Renombrar columnas |
| `drop()` | Eliminar filas/columnas |
| `groupby()` | Agrupar |
| `agg()` | Calcular múltiples agregaciones |
| `to_csv()` | Exportar a CSV |

## Idea central

> **Antes de crear un modelo de Machine Learning debemos asegurarnos de que los datos sean coherentes, completos y adecuados para el problema.**

La manipulación y limpieza con Pandas forma parte de la preparación necesaria para que el modelo aprenda de datos de buena calidad.

# Actividades para practicar (extras, no son obligatorias)

A partir de `df_limpio`, resolver:

1. Mostrar solamente los clientes de **Tierra del Fuego**.
2. Mostrar los clientes con **más de 10 compras anuales**.
3. Calcular el **ingreso promedio** general.
4. Obtener la cantidad de clientes **activos e inactivos**.
5. Calcular el ingreso promedio por `grupo_edad`.
6. Encontrar la provincia con mayor cantidad de clientes.
7. Crear una columna llamada `cliente_frecuente` que tenga `"Sí"` cuando `compras_anuales >= 15` y `"No"` en caso contrario.
8. Ordenar los clientes de mayor a menor según `satisfaccion`.
9. Explicar qué problemas tenía el dataset original.
10. Indicar cuáles de esos problemas podrían afectar a un modelo de Machine Learning y por qué.